In [ ]:
#import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import corner

In [ ]:
#read data

df = pd.read_csv('features_target.csv')

df = df.rename(columns={"Unnamed: 0": "Date"})
df = df.set_index("Date")
display(df.head())
display(df.tail())
#print(df.shape)

df.dtypes

In [ ]:
#check for missing values

missing_count = df.isnull().sum()
missing_pct = df.isnull().mean() * 100

missing_summary = pd.concat([missing_count, missing_pct], axis=1)
missing_summary.columns = ['missing_count', 'missing_pct']


missing_summary.to_csv("missing_values_summary.csv", index=True)


In [ ]:
# 僅使用 numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
df_numeric = df[numeric_cols]

# 計算 z-score
z_scores = np.abs(stats.zscore(df_numeric))

# 判斷是否為 outlier (Z > 3)
outliers_bool = z_scores > 3

# 記錄每筆資料的 outlier 變數
df['Outlier_Variables'] = [
    ", ".join(numeric_cols[outliers_bool[i]]) if outliers_bool[i].any() else ""
    for i in range(len(df))
]

# 抓出至少有一個 outlier 的 row
df_outliers = df[df['Outlier_Variables'] != ""]

print(f"總共有 {df_outliers.shape[0]} 筆 outlier row")
print(df_outliers.head())

# 計算每個變數的 outlier 次數
outlier_counts = pd.DataFrame({
    'Variable': numeric_cols,
    'Outlier_Count': outliers_bool.sum(axis=0)
})

# 導出 CSV
outlier_counts.to_csv("outlier_summary.csv", index=False)

outlier_counts.to_latex("outlier_summary.tex", index=False, caption="Outlier frequency for each variable", label="tab:outlier_summary")


# Boxplot
n_cols = 4
n_rows = int(np.ceil(len(numeric_cols)/n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(x=df[col], ax=axes[i])
    axes[i].set_title(col)

# 隱藏多餘的 subplot
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
'''
# 計算 Z-score（只針對 numeric columns）
numeric_df = df.select_dtypes(include=[np.number])
z_scores = np.abs(stats.zscore(numeric_df))
outliers = (z_scores > 3)

numeric_cols = numeric_df.columns

# 逐一針對每個欄位存 CSV
for i, col in enumerate(numeric_cols):
    df_outliers = df[outliers[:, i]]   # 取出該欄位的 outlier rows
    if not df_outliers.empty:          # 如果有 outlier 才存檔
        filename = f"outliers_{col}.csv"
        df_outliers.to_csv(filename)
        print(f"Saved：{filename} ({df_outliers.shape[0]} 筆)")
'''

In [ ]:
# 只針對 numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# 建立 summary table
summary = pd.DataFrame({
    "mean": numeric_df.mean(),
    "std": numeric_df.std(),
    "min": numeric_df.min(),
    "max": numeric_df.max(),
    "skewness": numeric_df.skew(),
    "kurtosis": numeric_df.kurtosis()
})

# 輸出到 CSV
summary.to_csv("summary_statistics.csv")
print("✅ Summary statistics saved to summary_statistics.csv")
print(summary)


summary.to_latex("summary statistics.tex", index=True, caption="Summary statistics", label="tab:summary_statistics")

In [ ]:
# look for variables with skewness > 1 or < -1

high_skew_cols = summary[(summary["skewness"].abs() > 1)].index.tolist()

print("📊 Highly skewed variables (|skewness| > 1):")
for col in high_skew_cols:
    print(f"- {col}: skewness = {summary.loc[col, 'skewness']:.2f}")


In [ ]:
# Histograms for numeric columns
for col in numeric_df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(numeric_df[col], bins=100, kde=True)  # kde加上平滑曲線更容易看偏態
    plt.title(f'Histogram of {col}')
    plt.show()


In [ ]:
# log transformation

# 要做 log transform 的變數
log_vars = ["SP500 30 Day Volatility", "SPX Put Volume", "Total SPX Options Volume", "VIX"]

# 建立一份副本
numeric_log = numeric_df.copy()

# 取 log（因為你已確認 min > 1，直接取 np.log 即可）
for col in log_vars:
    numeric_log[f"{col}_log"] = np.log(numeric_log[col])
    print(f"✅ Log-transformed {col}")

# 輸出成新檔案
numeric_log.to_csv("numeric_log_transformed.csv")
print("📂 Log-transformed dataset saved to numeric_log_transformed.csv")


In [ ]:
# 只針對 numeric columns
numeric_df_log = numeric_log.select_dtypes(include=[np.number])

# 建立 summary table
log_summary = pd.DataFrame({
    "mean": numeric_df_log.mean(),
    "std": numeric_df_log.std(),
    "min": numeric_df_log.min(),
    "max": numeric_df_log.max(),
    "skewness": numeric_df_log.skew(),
    "kurtosis": numeric_df_log.kurtosis()
})

# 輸出到 CSV
log_summary.to_csv("log_summary_statistics.csv")
print("✅ Summary statistics saved to summary_statistics.csv")
print(log_summary)

In [ ]:
# summary after Z-score
summary_log = pd.DataFrame({
    "mean": numeric_log.mean(),
    "std": numeric_log.std(),
    "min": numeric_log.min(),
    "max": numeric_log.max(),
    "skewness": numeric_log.skew(),
    "kurtosis": numeric_log.kurtosis()
})

summary_log.to_csv("summary_numeric_log.csv")
print(summary_log)

In [ ]:
log_vars = ["SP500 30 Day Volatility", "SPX Put Volume", "Total SPX Options Volume", "VIX"]

n_vars = len(log_vars)
fig, axes = plt.subplots(n_vars, 2, figsize=(12, n_vars*3))  # 每行一個變數，左右對照

for i, col in enumerate(log_vars):
    # 原始分布
    sns.histplot(numeric_df[col], bins=30, kde=True, ax=axes[i,0], color="skyblue")
    skew_orig = numeric_df[col].skew()
    axes[i,0].set_title(f"Original {col}")
    axes[i,0].set_xlabel(col)
    axes[i,0].set_ylabel("Frequency")
    axes[i,0].text(0.95, 0.95, f"Skewness = {skew_orig:.2f}", 
                   horizontalalignment='right', verticalalignment='top', 
                   transform=axes[i,0].transAxes, fontsize=10, bbox=dict(facecolor='white', alpha=0.6))

    # log 變換分布
    sns.histplot(numeric_df_log[f"{col}_log"], bins=30, kde=True, ax=axes[i,1], color="salmon")
    skew_log = numeric_df_log[f"{col}_log"].skew()
    axes[i,1].set_title(f"Log-Transformed {col}")
    axes[i,1].set_xlabel(f"log({col})")
    axes[i,1].set_ylabel("Frequency")
    axes[i,1].text(0.95, 0.95, f"Skewness = {skew_log:.2f}", 
                   horizontalalignment='right', verticalalignment='top', 
                   transform=axes[i,1].transAxes, fontsize=10, bbox=dict(facecolor='white', alpha=0.6))

plt.tight_layout()
plt.savefig("all_log_transform_comparison.png", dpi=300)
plt.show()

In [ ]:
zscore_scaler = StandardScaler()

# fit_transform 並保持欄位名稱和 index
numeric_zscore = pd.DataFrame(
    zscore_scaler.fit_transform(numeric_log),
    columns=numeric_log.columns,
    index=numeric_log.index
)

# 輸出 CSV
numeric_zscore.to_csv("numeric_log_zscore_scaled.csv")
print("Z-score standardized CSV saved as numeric_log_zscore_scaled.csv")

In [ ]:
# summary after Z-score
summary_zscore = pd.DataFrame({
    "mean": numeric_zscore.mean(),
    "std": numeric_zscore.std(),
    "min": numeric_zscore.min(),
    "max": numeric_zscore.max(),
    "skewness": numeric_zscore.skew(),
    "kurtosis": numeric_zscore.kurtosis()
})

summary_zscore.to_csv("summary_numeric_log_zscore_scaled.csv")
summary_zscore.to_latex("summary_numeric_log_zscore_scaled.tex", index=True, caption="Summary statistics after Z-score standardization", label="tab:summary_zscore_statistics")
print("Summary of Z-score standardized data saved")
print(summary_zscore)


In [ ]:
sns.pairplot(numeric_zscore) 

In [ ]:
# 假設你的 z-score 資料叫 numeric_zscore
df = numeric_zscore.copy()

# 目標變數
y_col = df.columns[1]  # 第二 column
y = df[y_col]

# 自變數 (除了目標變數)
x_cols = df.columns.drop(y_col)

# 逐一畫 scatter plot
for col in x_cols:
    plt.figure(figsize=(6,4))
    plt.scatter(df[col], y, alpha=0.5)
    plt.xlabel(col)
    plt.ylabel(y_col)
    plt.title(f"{col} vs {y_col}")
    plt.grid(True)
    plt.show()

In [ ]:
# 計算相關係數矩陣
corr_matrix = numeric_zscore.corr()

# 畫 heatmap
plt.figure(figsize=(24, 20))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Heatmap")
plt.show()